<a href="https://colab.research.google.com/github/shrisha-rao/text_classifier/blob/main/notebooks/Demo_Usage.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Setup

In [1]:
!git clone https://github.com/shrisha-rao/text_classifier.git

Cloning into 'text_classifier'...
remote: Enumerating objects: 150, done.
remote: Counting objects: 100% (150/150), done.
remote: Compressing objects: 100% (112/112), done.
remote: Total 150 (delta 89), reused 95 (delta 37), pack-reused 0 (from 0)
Receiving objects: 100% (150/150), 4.88 MiB | 7.47 MiB/s, done.
Resolving deltas: 100% (89/89), done.


In [2]:
%cd /content/text_classifier

/content/text_classifier


In [3]:
!git pull

Already up to date.


# HF Auth

In [4]:
from huggingface_hub import login, upload_folder
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


# Load BiEncoder model from HF

In [6]:
from model import BiEncoderModel
from transformers import AutoTokenizer
import torch
repo_name = "sraob/my_bi_encoder"
model = BiEncoderModel.from_pretrained(repo_name)
#
model.to("cuda" if torch.cuda.is_available() else "cpu");

Resolving sraob/my_bi_encoder from hf...


Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding and first 8 forzen


## Run Inference

In [7]:
texts = ["I love machine learning.", "Deep learning models are powerful."]
batch_labels = [
          ["AI", "Finance", 'Politics'],
          ["Deep Learning", "Neural Networks", "Agriculture"]
      ]
bi_output = model.forward_predict(texts, batch_labels)
bi_output

[{'text': 'I love machine learning.',
  'scores': {'AI': 0.94, 'Finance': 0.72, 'Politics': 0.49}},
 {'text': 'Deep learning models are powerful.',
  'scores': {'Deep Learning': 0.97,
   'Neural Networks': 1.0,
   'Agriculture': 0.35}}]

# Load PolyEncoder

In [8]:
from polyencoder import PolyencoderModel
from transformers import AutoTokenizer
import torch
repo_name = "sraob/my_poly_enc"
poly_model = PolyencoderModel.from_pretrained(repo_name)
#
poly_model.to("cuda" if torch.cuda.is_available() else "cpu");

polyencoder.pt:   0%|          | 0.00/438M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/796 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding and first 8 forzen


tokenizer_config.json:   0%|          | 0.00/322 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/712k [00:00<?, ?B/s]

In [9]:
texts = ["I love machine learning.", "Deep learning models are powerful."]
batch_labels = [
          ["AI", "Finance", 'Politics'],
          ["Deep Learning", "Neural Networks", "Agriculture"]
      ]
poly_output = poly_model.forward_predict(texts, batch_labels)
poly_output

[{'text': 'I love machine learning.',
  'scores': {'AI': 0.96, 'Finance': 0.5, 'Politics': 0.47}},
 {'text': 'Deep learning models are powerful.',
  'scores': {'Deep Learning': 0.98,
   'Neural Networks': 1.0,
   'Agriculture': 0.38}}]

# LLM as judge

In [10]:
!ls scripts

benchmark.py  generate_synthetic_data.py  llm_as_judge.py  train.py  utils.py


In [11]:
from scripts.llm_as_judge import llm_as_judge

In [19]:
from google.colab import userdata
import os
open_apikey = userdata.get('OPEN_API_KEY')
os.environ["OPENAI_API_KEY"] = open_apikey

In [20]:
llm_as_judge(bi_output, openai_api_key=open_apikey)

[{'text': 'I love machine learning.',
  'scores': {'AI': 0.94, 'Finance': 0.72, 'Politics': 0.49},
  'llm_score': 3,
  'justification': "The label 'AI' is highly relevant to the text and is correctly given a high score. However, 'Finance' and 'Politics' are not relevant to the text, yet they have moderate scores, which reduces the overall accuracy of the predictions."},
 {'text': 'Deep learning models are powerful.',
  'scores': {'Deep Learning': 0.97,
   'Neural Networks': 1.0,
   'Agriculture': 0.35},
  'llm_score': 4,
  'justification': "The predicted scores for 'Deep Learning' and 'Neural Networks' are highly relevant to the text, as both terms are closely related to the topic of deep learning models. However, the score for 'Agriculture' is less relevant, which slightly reduces the overall accuracy of the predictions."}]